> <div style="padding:15px 10px 1px 10px; border-radius:10px;"><h3> Early Depression Detection in Students </h3></div>

In [1]:
print("hello world")

hello world


> <p style="padding:10px;"> Fetching data from dataSet </p>

In [40]:
import pandas as pd

df = pd.read_csv("student_depression_dataset.csv")
df.head()

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,'5-6 hours',Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,'5-6 hours',Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,'Less than 5 hours',Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,'7-8 hours',Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,'5-6 hours',Moderate,M.Tech,Yes,1.0,1.0,No,0


> <p style="padding:10px;">IMPORTING LIBRARIES</p>

In [ ]:
from sklearn.model_selection import train_test_split
import PMeasures as Pm
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

print("done")

done


> <p style="padding:10px;">PreProcessing</p>

In [35]:
import pickle

preProcressedData = df.copy() # making copy of actual dataframe, so that the changes done in the dataframe dont cause any conflict in ACTUAL data

preProcressedData = preProcressedData.drop(columns=['Work Pressure','Profession','Job Satisfaction']) # dropping work pressure as it has 27898 0s and 3 5s (high bais)

l1 = ['Saanvi','M.Tech','Bhavna',"'Less Delhi'",'City','3.0',"'Less than 5 Kalyan'",'Mira','Harsha','Vaanya','Gaurav','Harsh','Reyansh','Kibara','Rashi','ME','M.Com','Nalyan','Mihir','Nalini','Nandini',]

preProcressedData = preProcressedData.drop(preProcressedData[preProcressedData['City'].isin(l1) ].index)

bin_cols = ['Gender','Have you ever had suicidal thoughts ?','Family History of Mental Illness'] # collection of all binary features

# encoding non binary values to binary values 

encoders = {}

for i in bin_cols:
    le = LabelEncoder() 
    preProcressedData[i] = le.fit_transform(preProcressedData[i])
    encoders[i] = le

preProcressedData = pd.get_dummies(preProcressedData,drop_first=True)

# saving encoded data columns in a Binary File

train_cols = preProcressedData.columns

with open("train_cols.pkl","wb") as fh:
    pickle.dump(train_cols,fh)

with open("label_encds.pkl","wb") as fh:
    pickle.dump(encoders,fh)
    
print("done")


done


> <p style="padding : 10px;">Performing Actual Training (decision tree), Train Test Split and measuring Performance</p> 

In [36]:
import math

X = preProcressedData.drop('Depression' , axis=1)
y = preProcressedData['Depression']

features = list(preProcressedData.columns)

def train_and_preformance(ratio):
    X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=ratio,random_state=0)
    
    DecisionModel = DecisionTreeClassifier(random_state=0,max_depth=5)
    DecisionModel.fit(X_train,y_train) # training
    
    y_pred = DecisionModel.predict(X_test)
        
    cM = confusion_matrix(y_test,y_pred)
    TN,FP,FN,TP = cM.ravel()
 
    
    return {
        "model":DecisionModel,
        "Accuracy":Pm.accuracy(TN,FP,FN,TP),
        "precision":Pm.precision(TN,FP,FN,TP),
        "recall":Pm.recall(TN,FP,FN,TP),
        "f1":Pm.fmeasure(TN,FP,FN,TP),
        "Specificity":Pm.specificity(TN,FP,FN,TP),
        "G Mean":Pm.gmean(TN,FP,FN,TP),
        "G Measure":Pm.gmeasure(TN,FP,FN,TP),
        "confusion matrix":cM,
    }
    
splitResult_90_10 = train_and_preformance(0.1)
splitResult_70_30 = train_and_preformance(0.3)

model = splitResult_70_30["model"]

print("Results of 90-10 split\n")
print("Accuracy\t\tPrecision\t\tRecall\t\t\tSpecificity\t\tf1Score\t\t\tG Mean\t\t\tG Measure")
print(f"{splitResult_90_10['Accuracy']}\t{splitResult_90_10['precision']}\t{splitResult_90_10['recall']}\t{splitResult_90_10['Specificity']}\t{splitResult_90_10['f1']}\t{splitResult_90_10['G Mean']}\t{splitResult_90_10['G Measure']}\t\n")
print(splitResult_90_10['confusion matrix'])
print()
print("\nResults of 70-30 split\n")
print("Accuracy\t\tPrecision\t\tRecall\t\t\tSpecificity\t\tf1Score\t\t\tG Mean\t\t\tG Measure")
print(f"{splitResult_70_30['Accuracy']}\t{splitResult_70_30['precision']}\t{splitResult_70_30['recall']}\t{splitResult_70_30['Specificity']}\t{splitResult_70_30['f1']}\t{splitResult_70_30['G Mean']}\t{splitResult_70_30['G Measure']}\t\n")
print(splitResult_70_30['confusion matrix'])


Results of 90-10 split

Accuracy		Precision		Recall			Specificity		f1Score			G Mean			G Measure
0.8170731707317073	0.8388278388278388	0.8481481481481481	0.15653775322283625	0.8434622467771637	0.3643723446026336	0.8434751200343402	

[[ 904  264]
 [ 246 1374]]


Results of 70-30 split

Accuracy		Precision		Recall			Specificity		f1Score			G Mean			G Measure
0.8218342699988043	0.8424097346898065	0.8578102782855982	0.1499597423510466	0.8500402576489534	0.3586600177295873	0.8500751313529645	

[[2650  790]
 [ 700 4223]]


> <p style="padding:10px">Predict on New value</p>

In [41]:
def predictOnNew():
    new_df = pd.read_csv("New_tests.csv")
    new_df2 = new_df.copy()
    new_df2 = new_df2.drop(columns=['Work Pressure'])
    
    with open("train_cols.pkl","rb") as fh:
        train_cols = pickle.load(fh)

    train_cols = train_cols.drop('Depression')

    with open("label_encds.pkl","rb") as fh:
        encoders = pickle.load(fh)
    
    for col,le in encoders.items():
        new_df2[col] = le.transform(new_df2[col])
        
    new_df2 = pd.get_dummies(new_df2,drop_first=True)
    new_df2 = new_df2.reindex(columns=train_cols,fill_value=0)
    
    pred = model.predict(new_df2)
    print(pred)
    return pred 

predictOnNew()
   

[1 1 0]


array([1, 1, 0])

> <p style="padding:10px">Visualization</p>

In [39]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20,10),dpi=1000)
plot_tree(model,feature_names=features,filled=True)
plt.show()